# UP Gram Panchayat Scraper (Colab)

This notebook installs dependencies and runs the scraper for:
- Gram Panchayat Head (post type 5)
- Gram Panchayat Member (post type 6)

It supports resume mode for long runs.

## 1) Mount Google Drive (recommended for long runs)
Outputs/checkpoint are saved in Drive so you can resume if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Configure repository URL
Replace `YOUR_GITHUB_REPO_URL` with your repo URL.

In [ ]:
REPO_URL = 'YOUR_GITHUB_REPO_URL'  # e.g. https://github.com/yourname/scrape.git
WORKDIR = '/content/drive/MyDrive/scrape'
OUTPUT_DIR = '/content/drive/MyDrive/scrape/2021_elections/output'

print('Repo:', REPO_URL)
print('Workdir:', WORKDIR)
print('Output:', OUTPUT_DIR)

## 3) Clone or update repo

In [ ]:
import os, subprocess, textwrap

if not os.path.exists(WORKDIR):
    !git clone {REPO_URL} {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull

%cd {WORKDIR}
!pwd

## 4) Install dependencies

In [ ]:
%cd {WORKDIR}
!pip install -r requirements.txt
!playwright install chromium
!playwright install-deps chromium

## 5) (Optional) Smoke test first
This tests only 5 gram panchayats.

In [ ]:
%cd {WORKDIR}
!python 2021_elections/scrape_gp_member_head_2021.py --headless --max-gp 5 --output-dir {OUTPUT_DIR}

## 6) Full run with resume
Run this cell multiple times whenever needed. It continues from `scrape_progress.json`.

In [ ]:
%cd {WORKDIR}
!python 2021_elections/scrape_gp_member_head_2021.py --headless --resume --output-dir {OUTPUT_DIR}

## 7) Check progress and files

In [ ]:
import os, json
from pathlib import Path

p = Path(OUTPUT_DIR)
print('Exists:', p.exists())
if p.exists():
    for f in sorted(p.glob('*')):
        print(f.name, '-', f.stat().st_size, 'bytes')

progress = p / 'scrape_progress.json'
if progress.exists():
    print('\nCheckpoint:')
    print(progress.read_text(encoding='utf-8'))

## 8) Download combined CSV

In [ ]:
from google.colab import files
files.download(f'{OUTPUT_DIR}/gp_member_head_candidates_2021.csv')